In [15]:
from prefect import flow, task
import requests
import pandas as pd
from pathlib import Path
import pandera as pa
from pandera import Column, DataFrameSchema, Check

# Importamos tu función común
from common.loaders import load_valid_country_ids

API_URL = "https://restcountries.com/v3.1/all?fields=cca3,borders"

# --- Definir el esquema dinámicamente con los IDs válidos ---
valid_country_ids = load_valid_country_ids()

borders_schema = DataFrameSchema({
    "id_country": Column(
        str,
        checks=[
            Check.str_length(3, 3),
            Check.isin(valid_country_ids)  # debe existir en countries.csv
        ],
        nullable=False
    ),
    "id_border_country": Column(
        str,
        checks=[
            Check.str_length(3, 3),
            Check.isin(valid_country_ids)  # también debe existir
        ],
        nullable=False
    ),
})


@task
def extract_borders():
    response = requests.get(API_URL)
    response.raise_for_status()
    return response.json()


@task
def transform_borders(data):
    df = pd.DataFrame(data)

    # Explode borders en filas
    df_exploded = df.explode("borders")

    # Renombrar columnas
    df_exploded = df_exploded.rename(
        columns={"cca3": "id_country", "borders": "id_border_country"}
    )

    # Quitar filas sin fronteras
    df_exploded = df_exploded.dropna(subset=["id_border_country"])

    # Eliminar duplicados
    df_exploded = df_exploded.drop_duplicates()

    return df_exploded.reset_index(drop=True)


@task
def validate_borders(df: pd.DataFrame) -> pd.DataFrame:
    return borders_schema.validate(df)


@task
def load_borders(df: pd.DataFrame):
    file_path = Path("../stage/country_borders.csv")
    file_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(file_path, index=False, encoding="utf-8")
    print(f" Saved {len(df)} rows to {file_path}")


@flow(name="etl_borders_flow")
def etl_borders():
    data = extract_borders()
    df = transform_borders(data)
    df = validate_borders(df)
    load_borders(df)


if __name__ == "__main__":
    etl_borders()


03:25:24.448 | INFO    | Flow run 'psychedelic-iguana' - Beginning flow run 'psychedelic-iguana' for flow 'etl_borders_flow'

03:25:25.448 | INFO    | Task run 'extract_borders-c8a' - Finished in state Completed()

03:25:25.682 | INFO    | Task run 'transform_borders-f3d' - Finished in state Completed()

03:25:25.906 | INFO    | Task run 'validate_borders-e0f' - Finished in state Completed()

✅ Saved 649 rows to ..\stage\country_borders.csv


03:25:26.124 | INFO    | Task run 'load_borders-800' - Finished in state Completed()

03:25:26.137 | INFO    | Flow run 'psychedelic-iguana' - Finished in state Completed()